
# TrackPulse — Silver Layer Transformation

This notebook transforms raw Bronze datasets into structured, validated Silver datasets.

The Silver layer parses raw records, standardizes data types, checks data quality, and prepares datasets for downstream analytics.




## 1. Telemetry — Parsing and Data Quality

Parse raw telemetry JSON records into structured columns, convert data types, and validate essential fields before saving the Silver dataset.


In [0]:

# Read raw telemetry from Bronze and parse each JSON record into a structured DataFrame.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, LongType
)

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

bronze_telemetry_path = storage_path + "bronze/telemetry/"
silver_telemetry_path = storage_path + "silver/telemetry/"

bronze_telemetry_df = (
    spark.read.format("delta")
    .load(bronze_telemetry_path)
)

display(bronze_telemetry_df.select("value").limit(5))


value
"{""event_id"":""VEH-00001-0000840"",""event_timestamp"":""2026-10-01T14:00:00Z"",""vehicle_id"":""VEH-00001"",""latitude"":30.01485,""longitude"":31.382751,""speed_kph"":51.54,""odometer_km"":158465.127,""rpm"":2128,""fuel_rate_lph"":4.412,""engine_temp_c"":85.53,""ignition_on"":true,""trip_id"":""TRIP-00001-001"",""shipment_id"":""SHP-00001"",""driver_id"":""DRV-00001"",""route_id"":""RTE-001"",""cargo_temp_c"":null}"
"{""event_id"":""VEH-00002-0000840"",""event_timestamp"":""2026-10-01T14:00:00Z"",""vehicle_id"":""VEH-00002"",""latitude"":30.148734,""longitude"":31.217599,""speed_kph"":40.64,""odometer_km"":47670.716,""rpm"":1797,""fuel_rate_lph"":6.191,""engine_temp_c"":95.0,""ignition_on"":true,""trip_id"":""TRIP-00002-001"",""shipment_id"":""SHP-00002"",""driver_id"":""DRV-00001"",""route_id"":""RTE-010"",""cargo_temp_c"":null}"
"{""event_id"":""VEH-00003-0000840"",""event_timestamp"":""2026-10-01T14:00:00Z"",""vehicle_id"":""VEH-00003"",""latitude"":29.997252,""longitude"":31.170903,""speed_kph"":40.81,""odometer_km"":40982.187,""rpm"":2435,""fuel_rate_lph"":8.705,""engine_temp_c"":96.63,""ignition_on"":true,""trip_id"":""TRIP-00003-001"",""shipment_id"":""SHP-00003"",""driver_id"":""DRV-00001"",""route_id"":""RTE-001"",""cargo_temp_c"":null}"
"{""event_id"":""VEH-00004-0000840"",""event_timestamp"":""2026-10-01T14:00:00Z"",""vehicle_id"":""VEH-00004"",""latitude"":29.137195,""longitude"":31.109148,""speed_kph"":33.69,""odometer_km"":173325.555,""rpm"":1644,""fuel_rate_lph"":5.618,""engine_temp_c"":87.52,""ignition_on"":true,""trip_id"":""TRIP-00004-001"",""shipment_id"":""SHP-00004"",""driver_id"":""DRV-00002"",""route_id"":""RTE-007"",""cargo_temp_c"":null}"
"{""event_id"":""VEH-00005-0000840"",""event_timestamp"":""2026-10-01T14:00:00Z"",""vehicle_id"":""VEH-00005"",""latitude"":29.086538,""longitude"":31.10219,""speed_kph"":29.49,""odometer_km"":377956.855,""rpm"":1544,""fuel_rate_lph"":3.766,""engine_temp_c"":91.29,""ignition_on"":true,""trip_id"":""TRIP-00005-001"",""shipment_id"":""SHP-00005"",""driver_id"":""DRV-00002"",""route_id"":""RTE-007"",""cargo_temp_c"":null}"


In [0]:

# Define the telemetry schema and parse raw JSON records into typed columns.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, LongType, BooleanType
)

telemetry_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("speed_kph", DoubleType(), True),
    StructField("odometer_km", DoubleType(), True),
    StructField("rpm", LongType(), True),
    StructField("fuel_rate_lph", DoubleType(), True),
    StructField("engine_temp_c", DoubleType(), True),
    StructField("ignition_on", BooleanType(), True),
    StructField("trip_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("route_id", StringType(), True),
    StructField("cargo_temp_c", DoubleType(), True)
])

# Read the raw Bronze dataset.
bronze_telemetry_df = (
    spark.read.format("delta")
    .load(bronze_telemetry_path)
)

# Parse each JSON string using the explicit schema.
silver_telemetry_df = (
    bronze_telemetry_df
    .withColumn(
        "parsed",
        F.from_json(F.col("value"), telemetry_schema)
    )
    .select(
        "parsed.*",
        "source_file",
        "ingested_at"
    )
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            "event_timestamp",
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )
)

print("Parsed telemetry schema:")
silver_telemetry_df.printSchema()

display(silver_telemetry_df.limit(5))


Parsed telemetry schema:
root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- speed_kph: double (nullable = true)
 |-- odometer_km: double (nullable = true)
 |-- rpm: long (nullable = true)
 |-- fuel_rate_lph: double (nullable = true)
 |-- engine_temp_c: double (nullable = true)
 |-- ignition_on: boolean (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- cargo_temp_c: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



event_id,event_timestamp,vehicle_id,latitude,longitude,speed_kph,odometer_km,rpm,fuel_rate_lph,engine_temp_c,ignition_on,trip_id,shipment_id,driver_id,route_id,cargo_temp_c,source_file,ingested_at
VEH-00001-0000980,2026-10-01T16:20:00.000Z,VEH-00001,30.011029,31.203936,36.69,158586.75,2011,7.42,85.25,true,TRIP-00001-001,SHP-00001,DRV-00001,RTE-001,null,telemetry_000099.jsonl,2026-10-26T22:45:46.984Z
VEH-00002-0000980,2026-10-01T16:20:00.000Z,VEH-00002,30.166106,31.2419,54.89,47789.486,2136,7.662,95.66,true,TRIP-00002-001,SHP-00002,DRV-00001,RTE-010,null,telemetry_000099.jsonl,2026-10-26T22:45:46.984Z
VEH-00003-0000980,2026-10-01T16:20:00.000Z,VEH-00003,30.040386,31.232244,60.19,41130.994,2415,8.593,95.58,true,TRIP-00003-001,SHP-00003,DRV-00001,RTE-001,null,telemetry_000099.jsonl,2026-10-26T22:45:46.984Z
VEH-00004-0000980,2026-10-01T16:20:00.000Z,VEH-00004,29.274141,31.127943,39.2,173419.483,1834,6.128,88.58,true,TRIP-00004-001,SHP-00004,DRV-00002,RTE-007,null,telemetry_000099.jsonl,2026-10-26T22:45:46.984Z
VEH-00005-0000980,2026-10-01T16:20:00.000Z,VEH-00005,30.020417,31.232155,52.02,378071.026,1942,5.532,91.54,true,TRIP-00005-001,SHP-00005,DRV-00002,RTE-007,null,telemetry_000099.jsonl,2026-10-26T22:45:46.984Z


In [0]:

# Check parsing failures and essential fields without scanning the full dataset repeatedly.

quality_check_df = silver_telemetry_df.agg(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(F.col("event_id").isNull(), 1).otherwise(0)
    ).alias("null_event_ids"),
    F.sum(
        F.when(F.col("vehicle_id").isNull(), 1).otherwise(0)
    ).alias("null_vehicle_ids"),
    F.sum(
        F.when(F.col("event_timestamp").isNull(), 1).otherwise(0)
    ).alias("null_timestamps"),
    F.sum(
        F.when(F.col("latitude").isNull(), 1).otherwise(0)
    ).alias("null_latitudes"),
    F.sum(
        F.when(F.col("longitude").isNull(), 1).otherwise(0)
    ).alias("null_longitudes")
)

display(quality_check_df)


total_rows,null_event_ids,null_vehicle_ids,null_timestamps,null_latitudes,null_longitudes
10000000,0,0,0,0,0


In [0]:

# Save the parsed and typed telemetry records to the Silver Delta layer.

(
    silver_telemetry_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_telemetry_path)
)

print("Silver telemetry transformation completed!")


Silver telemetry transformation completed!



## 2. Anomaly Labels — Parsing and Data Quality

Parse raw anomaly JSON records into structured columns, preserve ingestion metadata, and validate essential fields.


In [0]:

# Read raw anomaly labels from Bronze and parse JSON into structured columns.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

bronze_anomaly_path = storage_path + "bronze/anomaly_labels/"
silver_anomaly_path = storage_path + "silver/anomaly_labels/"

anomaly_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("anomaly_type", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("details", StringType(), True),
    StructField("incident_id", StringType(), True)
])

bronze_anomaly_df = (
    spark.read.format("delta")
    .load(bronze_anomaly_path)
)

silver_anomaly_df = (
    bronze_anomaly_df
    .withColumn(
        "parsed",
        F.from_json(F.col("value"), anomaly_schema)
    )
    .select(
        "parsed.*",
        "source_file",
        "ingested_at"
    )
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            "event_timestamp",
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )
)

print("Parsed anomaly schema:")
silver_anomaly_df.printSchema()

display(silver_anomaly_df.limit(5))


Parsed anomaly schema:
root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- anomaly_type: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- details: string (nullable = true)
 |-- incident_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



event_id,event_timestamp,vehicle_id,anomaly_type,severity,details,incident_id,source_file,ingested_at
VEH-00009-0000370,2026-10-01T06:10:00.000Z,VEH-00009,harsh_braking,high,synthetic abrupt speed drop: 19.29 kph over 60s,null,anomalies_000038.jsonl,2026-10-26T23:34:13.871Z
VEH-00014-0000370,2026-10-01T06:10:00.000Z,VEH-00014,engine_overheat,critical,OBD-II engine_temp_c=116.09; synthetic overheat injected,null,anomalies_000038.jsonl,2026-10-26T23:34:13.871Z
VEH-00018-0000370,2026-10-01T06:10:00.000Z,VEH-00018,fuel_anomaly,high,OBD-II fuel_rate_lph=13.741; injected excess consumption,null,anomalies_000038.jsonl,2026-10-26T23:34:13.871Z
VEH-00021-0000370,2026-10-01T06:10:00.000Z,VEH-00021,excessive_idling,medium,speed_kph=0; ignition_on=True; rpm=852; elevated idle fuel consumption,null,anomalies_000038.jsonl,2026-10-26T23:34:13.871Z
VEH-00022-0000370,2026-10-01T06:10:00.000Z,VEH-00022,harsh_braking,high,synthetic abrupt speed drop: 27.58 kph over 60s,null,anomalies_000038.jsonl,2026-10-26T23:34:13.871Z


In [0]:

# Check anomaly record counts and essential fields in one aggregation.

anomaly_quality_df = silver_anomaly_df.agg(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(F.col("event_id").isNull(), 1).otherwise(0)
    ).alias("null_event_ids"),
    F.sum(
        F.when(F.col("vehicle_id").isNull(), 1).otherwise(0)
    ).alias("null_vehicle_ids"),
    F.sum(
        F.when(F.col("event_timestamp").isNull(), 1).otherwise(0)
    ).alias("null_timestamps"),
    F.sum(
        F.when(F.col("anomaly_type").isNull(), 1).otherwise(0)
    ).alias("null_anomaly_types"),
    F.sum(
        F.when(F.col("severity").isNull(), 1).otherwise(0)
    ).alias("null_severities")
)

display(anomaly_quality_df)


total_rows,null_event_ids,null_vehicle_ids,null_timestamps,null_anomaly_types,null_severities
1499620,0,0,0,0,0


In [0]:

# Save parsed anomaly labels to Silver after the initial quality checks pass.

(
    silver_anomaly_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_anomaly_path)
)

print("Silver anomaly labels transformation completed!")


Silver anomaly labels transformation completed!


## 3. Trips — Parsing and Data Quality

Parse the raw trips JSON array into individual trip records, standardize data types, and validate trip identifiers and timestamps.

In [0]:
# Inspect the raw trips JSON array stored in the Bronze layer.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, DoubleType
)

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

bronze_trips_path = storage_path + "bronze/trips/"
silver_trips_path = storage_path + "silver/trips/"

bronze_trips_df = spark.read.format("delta").load(bronze_trips_path)

value


In [0]:
# Display only the first 500 characters of the raw trips JSON.

display(
    bronze_trips_df
    .select(F.substring("value", 1, 500).alias("json_sample"))
    .limit(1)
)

json_sample
"[{""trip_id"":""TRIP-00001-001"",""vehicle_id"":""VEH-00001"",""driver_id"":""DRV-00001"",""shipment_id"":""SHP-00001"",""route_id"":""RTE-001"",""planned_start"":""2026-10-01T00:00:00Z"",""planned_end"":""2026-10-02T09:19:00Z"",""trip_status"":""in_progress"",""actual_start"":""2026-10-01T00:00:00Z"",""actual_end"":null,""distance_km"":null,""fuel_used_liters"":null,""origin"":""cairo_ring"",""destination"":[30.0444,31.2357]},{""trip_id"":""TRIP-00002-001"",""vehicle_id"":""VEH-00002"",""driver_id"":""DRV-00001"",""shipment_id"":""SHP-00002"",""route_id"":""RT"


In [0]:
# Define the schema and parse the JSON array into individual trip records.

from pyspark.sql.types import (
    StructType, StructField, StringType,
    ArrayType, DoubleType
)

trip_schema = StructType([
    StructField("trip_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("route_id", StringType(), True),
    StructField("planned_start", StringType(), True),
    StructField("planned_end", StringType(), True),
    StructField("trip_status", StringType(), True),
    StructField("actual_start", StringType(), True),
    StructField("actual_end", StringType(), True),
    StructField("distance_km", StringType(), True),
    StructField("fuel_used_liters", StringType(), True),
    StructField("origin", StringType(), True),
    StructField("destination", ArrayType(DoubleType()), True)
])

# Parse the JSON array and expand each trip into a separate row.
silver_trips_df = (
    bronze_trips_df
    .withColumn(
        "parsed_trips",
        F.from_json(F.col("value"), ArrayType(trip_schema))
    )
    .select(
        F.explode("parsed_trips").alias("trip"),
        "source_file",
        "ingested_at"
    )
    .select("trip.*", "source_file", "ingested_at")
)

# Convert timestamps and numeric fields to their appropriate types.
silver_trips_df = (
    silver_trips_df
    .withColumn(
        "planned_start",
        F.to_timestamp("planned_start", "yyyy-MM-dd'T'HH:mm:ssX")
    )
    .withColumn(
        "planned_end",
        F.to_timestamp("planned_end", "yyyy-MM-dd'T'HH:mm:ssX")
    )
    .withColumn(
        "actual_start",
        F.to_timestamp("actual_start", "yyyy-MM-dd'T'HH:mm:ssX")
    )
    .withColumn(
        "actual_end",
        F.to_timestamp("actual_end", "yyyy-MM-dd'T'HH:mm:ssX")
    )
    .withColumn("distance_km", F.col("distance_km").cast("double"))
    .withColumn("fuel_used_liters", F.col("fuel_used_liters").cast("double"))
)

# Show a small sample and the resulting schema.
silver_trips_df.printSchema()
display(silver_trips_df.limit(5))

root
 |-- trip_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- planned_start: timestamp (nullable = true)
 |-- planned_end: timestamp (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- actual_start: timestamp (nullable = true)
 |-- actual_end: timestamp (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fuel_used_liters: double (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



trip_id,vehicle_id,driver_id,shipment_id,route_id,planned_start,planned_end,trip_status,actual_start,actual_end,distance_km,fuel_used_liters,origin,destination,source_file,ingested_at
TRIP-00001-001,VEH-00001,DRV-00001,SHP-00001,RTE-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,cairo_ring,"List(30.0444, 31.2357)",trips.json,2026-10-26T23:40:50.754Z
TRIP-00002-001,VEH-00002,DRV-00001,SHP-00002,RTE-010,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,north_cairo,"List(30.0444, 31.2357)",trips.json,2026-10-26T23:40:50.754Z
TRIP-00003-001,VEH-00003,DRV-00001,SHP-00003,RTE-001,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,cairo_ring,"List(30.0444, 31.2357)",trips.json,2026-10-26T23:40:50.754Z
TRIP-00004-001,VEH-00004,DRV-00002,SHP-00004,RTE-007,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,beni_suef,"List(29.0661, 31.0994)",trips.json,2026-10-26T23:40:50.754Z
TRIP-00005-001,VEH-00005,DRV-00002,SHP-00005,RTE-007,2026-10-01T00:00:00.000Z,2026-10-02T09:19:00.000Z,in_progress,2026-10-01T00:00:00.000Z,null,null,null,beni_suef,"List(29.0661, 31.0994)",trips.json,2026-10-26T23:40:50.754Z


In [0]:
# Validate trip counts, unique identifiers, and essential fields.

trips_quality_df = silver_trips_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("trip_id").alias("unique_trip_ids"),
    F.sum(F.when(F.col("trip_id").isNull(), 1).otherwise(0)).alias("null_trip_ids"),
    F.sum(F.when(F.col("vehicle_id").isNull(), 1).otherwise(0)).alias("null_vehicle_ids"),
    F.sum(F.when(F.col("planned_start").isNull(), 1).otherwise(0)).alias("null_planned_starts"),
    F.sum(F.when(F.col("trip_status").isNull(), 1).otherwise(0)).alias("null_trip_statuses"),
    F.sum(F.when(F.col("destination").isNull(), 1).otherwise(0)).alias("null_destinations"),
    F.sum(F.when(F.col("actual_end").isNull(), 1).otherwise(0)).alias("missing_actual_ends"),
    F.sum(F.when(F.col("distance_km").isNull(), 1).otherwise(0)).alias("missing_distances"),
    F.sum(F.when(F.col("fuel_used_liters").isNull(), 1).otherwise(0)).alias("missing_fuel_values")
)

display(trips_quality_df)

total_rows,unique_trip_ids,null_trip_ids,null_vehicle_ids,null_planned_starts,null_trip_statuses,null_destinations,missing_actual_ends,missing_distances,missing_fuel_values
5000,5000,0,0,0,0,0,5000,5000,5000


In [0]:
# Save the parsed and validated trips to the Silver Delta layer.

(
    silver_trips_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_trips_path)
)

print("Silver trips transformation completed!")

Silver trips transformation completed!


## 4. Master Data — Vehicles

Parse raw vehicle records from Bronze, standardize data types, and validate vehicle identifiers before saving the Silver dataset.

In [0]:
# Inspect a small sample of raw vehicle records from Bronze.

bronze_vehicles_path = storage_path + "bronze/master_data/vehicles/"

bronze_vehicles_df = spark.read.format("delta").load(bronze_vehicles_path)

display(
    bronze_vehicles_df
    .select("value")
    .limit(5)
)

value
"vehicle_id,vehicle_type,model,manufacture_year,fuel_type,capacity_kg,status"
"VEH-00001,truck,Heavy Duty Truck,2018,diesel,12000,active"
"VEH-00002,truck,Heavy Duty Truck,2024,diesel,12000,active"
"VEH-00003,truck,Heavy Duty Truck,2022,diesel,12000,active"
"VEH-00004,truck,Heavy Duty Truck,2019,diesel,12000,active"


In [0]:
# Parse vehicle records from Bronze and standardize their data types.

vehicle_schema = """
    vehicle_id STRING,
    vehicle_type STRING,
    model STRING,
    manufacture_year STRING,
    fuel_type STRING,
    capacity_kg STRING,
    status STRING
"""

silver_vehicles_path = storage_path + "silver/master_data/vehicles/"

silver_vehicles_df = (
    bronze_vehicles_df
    # Remove the CSV header row before parsing.
    .filter(~F.col("value").startswith("vehicle_id,"))
    # Parse each raw CSV record into structured columns.
    .withColumn(
        "parsed",
        F.from_csv(
            F.col("value"),
            vehicle_schema,
            {"sep": ",", "mode": "PERMISSIVE"}
        )
    )
    .select("parsed.*", "source_file", "ingested_at")
    # Convert numeric fields to their proper types.
    .withColumn("manufacture_year", F.col("manufacture_year").cast("int"))
    .withColumn("capacity_kg", F.col("capacity_kg").cast("double"))
)

# Inspect the parsed schema and a small sample.
silver_vehicles_df.printSchema()
display(silver_vehicles_df.limit(5))

root
 |-- vehicle_id: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- model: string (nullable = true)
 |-- manufacture_year: integer (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- capacity_kg: double (nullable = true)
 |-- status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



vehicle_id,vehicle_type,model,manufacture_year,fuel_type,capacity_kg,status,source_file,ingested_at
VEH-00001,truck,Heavy Duty Truck,2018,diesel,12000.0,active,vehicles.csv,2026-10-26T23:48:04.418Z
VEH-00002,truck,Heavy Duty Truck,2024,diesel,12000.0,active,vehicles.csv,2026-10-26T23:48:04.418Z
VEH-00003,truck,Heavy Duty Truck,2022,diesel,12000.0,active,vehicles.csv,2026-10-26T23:48:04.418Z
VEH-00004,truck,Heavy Duty Truck,2019,diesel,12000.0,active,vehicles.csv,2026-10-26T23:48:04.418Z
VEH-00005,van,Transit Van,2023,diesel,1500.0,active,vehicles.csv,2026-10-26T23:48:04.418Z


In [0]:
# Check vehicle record counts, key uniqueness, and missing values.

display(
    silver_vehicles_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("vehicle_id").alias("unique_vehicle_ids"),
        F.sum(F.when(F.col("vehicle_id").isNull(), 1).otherwise(0)).alias("null_vehicle_ids"),
        F.sum(F.when(F.col("manufacture_year").isNull(), 1).otherwise(0)).alias("null_manufacture_years"),
        F.sum(F.when(F.col("capacity_kg").isNull(), 1).otherwise(0)).alias("null_capacities")
    )
)

total_rows,unique_vehicle_ids,null_vehicle_ids,null_manufacture_years,null_capacities
5000,5000,0,0,0


In [0]:
# Save validated vehicle records to the Silver Delta layer.

(
    silver_vehicles_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_vehicles_path)
)

print("Silver vehicles transformation completed!")

Silver vehicles transformation completed!


## 5. Drivers — Parsing and Data Quality

Parse raw driver records from Bronze, preserve source lineage, and validate driver identifiers.

In [0]:
# Inspect a small sample of raw driver records from Bronze.

bronze_drivers_path = storage_path + "bronze/master_data/drivers/"

bronze_drivers_df = spark.read.format("delta").load(bronze_drivers_path)

display(
    bronze_drivers_df
    .select("value")
    .limit(5)
)

value
"driver_id,driver_label,license_category,employment_status"
"DRV-00001,Driver 00001,professional_heavy,active"
"DRV-00002,Driver 00002,professional_heavy,active"
"DRV-00003,Driver 00003,professional_light,active"
"DRV-00004,Driver 00004,professional_heavy,active"


In [0]:
# Parse driver records from Bronze and preserve source metadata.

driver_schema = """
    driver_id STRING,
    driver_label STRING,
    license_category STRING,
    employment_status STRING
"""

silver_drivers_path = storage_path + "silver/master_data/drivers/"

silver_drivers_df = (
    bronze_drivers_df
    # Exclude the raw CSV header.
    .filter(~F.col("value").startswith("driver_id,"))
    # Convert each CSV line into structured columns.
    .withColumn(
        "parsed",
        F.from_csv(
            F.col("value"),
            driver_schema,
            {"sep": ",", "mode": "PERMISSIVE"}
        )
    )
    .select("parsed.*", "source_file", "ingested_at")
)

# Review the resulting schema and sample records.
silver_drivers_df.printSchema()
display(silver_drivers_df.limit(5))

root
 |-- driver_id: string (nullable = true)
 |-- driver_label: string (nullable = true)
 |-- license_category: string (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



driver_id,driver_label,license_category,employment_status,source_file,ingested_at
DRV-00001,Driver 00001,professional_heavy,active,drivers.csv,2026-10-26T23:45:15.080Z
DRV-00002,Driver 00002,professional_heavy,active,drivers.csv,2026-10-26T23:45:15.080Z
DRV-00003,Driver 00003,professional_light,active,drivers.csv,2026-10-26T23:45:15.080Z
DRV-00004,Driver 00004,professional_heavy,active,drivers.csv,2026-10-26T23:45:15.080Z
DRV-00005,Driver 00005,professional_heavy,active,drivers.csv,2026-10-26T23:45:15.080Z


In [0]:
# Check driver counts, unique IDs, and missing essential fields.

display(
    silver_drivers_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("driver_id").alias("unique_driver_ids"),
        F.sum(F.when(F.col("driver_id").isNull(), 1).otherwise(0)).alias("null_driver_ids"),
        F.sum(F.when(F.col("license_category").isNull(), 1).otherwise(0)).alias("null_license_categories"),
        F.sum(F.when(F.col("employment_status").isNull(), 1).otherwise(0)).alias("null_employment_statuses")
    )
)

total_rows,unique_driver_ids,null_driver_ids,null_license_categories,null_employment_statuses
1400,1400,0,0,0


In [0]:
# Save validated driver records to the Silver Delta layer.

(
    silver_drivers_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_drivers_path)
)

print("Silver drivers transformation completed!")

Silver drivers transformation completed!


## 6. Routes — Parsing and Data Quality

Parse raw route records from Bronze, standardize numeric fields, and validate route identifiers.

In [0]:
# Inspect raw route records before defining the parsing schema.

bronze_routes_path = storage_path + "bronze/master_data/routes/"

bronze_routes_df = spark.read.format("delta").load(bronze_routes_path)

display(
    bronze_routes_df
    .select("value")
    .limit(5)
)

value
"route_id,route_name,origin_name,destination_name,origin_latitude,origin_longitude,destination_latitude,destination_longitude,planned_distance_km,planned_duration_min,deviation_threshold_m,route_note"
"RTE-001,cairo_ring,Cairo,Cairo Ring,30.0444,31.2357,30.0444,31.2357,,,1500,Synthetic corridor/waypoints; not a routable road-network route"
"RTE-002,new_cairo,Cairo,New Cairo,30.0444,31.2357,30.0444,31.2357,,,1500,Synthetic corridor/waypoints; not a routable road-network route"
"RTE-003,alex_desert,Cairo,Alex Desert,30.0444,31.2357,31.2001,29.9187,,,1500,Synthetic corridor/waypoints; not a routable road-network route"
"RTE-004,ismailia,Cairo,Ismailia,30.0444,31.2357,30.5965,32.2715,,,1500,Synthetic corridor/waypoints; not a routable road-network route"


In [0]:
# Parse route records and convert numeric columns to appropriate types.

route_schema = """
    route_id STRING,
    route_name STRING,
    origin_name STRING,
    destination_name STRING,
    origin_latitude STRING,
    origin_longitude STRING,
    destination_latitude STRING,
    destination_longitude STRING,
    planned_distance_km STRING,
    planned_duration_min STRING,
    deviation_threshold_m STRING,
    route_note STRING
"""

silver_routes_path = storage_path + "silver/master_data/routes/"

silver_routes_df = (
    bronze_routes_df
    # Remove the CSV header before parsing the records.
    .filter(~F.col("value").startswith("route_id,"))
    .withColumn(
        "parsed",
        F.from_csv(
            F.col("value"),
            route_schema,
            {"sep": ",", "mode": "PERMISSIVE"}
        )
    )
    .select("parsed.*", "source_file", "ingested_at")
)

# Cast coordinates, planned metrics, and deviation threshold to numeric types.
double_columns = [
    "origin_latitude",
    "origin_longitude",
    "destination_latitude",
    "destination_longitude",
    "planned_distance_km"
]

for column_name in double_columns:
    silver_routes_df = silver_routes_df.withColumn(
        column_name, F.col(column_name).cast("double")
    )

silver_routes_df = (
    silver_routes_df
    .withColumn("planned_duration_min", F.col("planned_duration_min").cast("int"))
    .withColumn("deviation_threshold_m", F.col("deviation_threshold_m").cast("double"))
)

# Inspect the structured schema and a small sample.
silver_routes_df.printSchema()
display(silver_routes_df.limit(5))

root
 |-- route_id: string (nullable = true)
 |-- route_name: string (nullable = true)
 |-- origin_name: string (nullable = true)
 |-- destination_name: string (nullable = true)
 |-- origin_latitude: double (nullable = true)
 |-- origin_longitude: double (nullable = true)
 |-- destination_latitude: double (nullable = true)
 |-- destination_longitude: double (nullable = true)
 |-- planned_distance_km: double (nullable = true)
 |-- planned_duration_min: integer (nullable = true)
 |-- deviation_threshold_m: double (nullable = true)
 |-- route_note: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



route_id,route_name,origin_name,destination_name,origin_latitude,origin_longitude,destination_latitude,destination_longitude,planned_distance_km,planned_duration_min,deviation_threshold_m,route_note,source_file,ingested_at
RTE-001,cairo_ring,Cairo,Cairo Ring,30.0444,31.2357,30.0444,31.2357,null,null,1500.0,Synthetic corridor/waypoints; not a routable road-network route,routes.csv,2026-10-26T23:48:01.220Z
RTE-002,new_cairo,Cairo,New Cairo,30.0444,31.2357,30.0444,31.2357,null,null,1500.0,Synthetic corridor/waypoints; not a routable road-network route,routes.csv,2026-10-26T23:48:01.220Z
RTE-003,alex_desert,Cairo,Alex Desert,30.0444,31.2357,31.2001,29.9187,null,null,1500.0,Synthetic corridor/waypoints; not a routable road-network route,routes.csv,2026-10-26T23:48:01.220Z
RTE-004,ismailia,Cairo,Ismailia,30.0444,31.2357,30.5965,32.2715,null,null,1500.0,Synthetic corridor/waypoints; not a routable road-network route,routes.csv,2026-10-26T23:48:01.220Z
RTE-005,suez,Cairo,Suez,30.0444,31.2357,29.9668,32.5498,null,null,1500.0,Synthetic corridor/waypoints; not a routable road-network route,routes.csv,2026-10-26T23:48:01.220Z


In [0]:
# Validate route counts, identifiers, coordinates, and source-null metrics.

display(
    silver_routes_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("route_id").alias("unique_route_ids"),
        F.sum(F.when(F.col("route_id").isNull(), 1).otherwise(0)).alias("null_route_ids"),
        F.sum(F.when(F.col("origin_latitude").isNull(), 1).otherwise(0)).alias("null_origin_latitudes"),
        F.sum(F.when(F.col("destination_latitude").isNull(), 1).otherwise(0)).alias("null_destination_latitudes"),
        F.sum(F.when(F.col("planned_distance_km").isNull(), 1).otherwise(0)).alias("null_planned_distances"),
        F.sum(F.when(F.col("planned_duration_min").isNull(), 1).otherwise(0)).alias("null_planned_durations")
    )
)

total_rows,unique_route_ids,null_route_ids,null_origin_latitudes,null_destination_latitudes,null_planned_distances,null_planned_durations
10,10,0,0,0,10,10


In [0]:
# Save validated route records to the Silver Delta layer.

(
    silver_routes_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_routes_path)
)

print("Silver routes transformation completed!")

Silver routes transformation completed!


## 7. Shipments — Parsing and Data Quality

Parse shipment records from Bronze, standardize required temperature limits, and validate shipment identifiers.

In [0]:
# Inspect a small sample of raw shipment records from Bronze.

bronze_shipments_path = storage_path + "bronze/master_data/shipments/"

bronze_shipments_df = spark.read.format("delta").load(bronze_shipments_path)

display(
    bronze_shipments_df
    .select("value")
    .limit(5)
)

value
"shipment_id,cargo_type,required_temp_min_c,required_temp_max_c,delivery_status"
"SHP-00001,general_goods,,,planned"
"SHP-00002,general_goods,,,planned"
"SHP-00003,general_goods,,,planned"
"SHP-00004,general_goods,,,planned"


In [0]:
# Parse shipment records and standardize temperature limit data types.

shipment_schema = """
    shipment_id STRING,
    cargo_type STRING,
    required_temp_min_c STRING,
    required_temp_max_c STRING,
    delivery_status STRING
"""

silver_shipments_path = storage_path + "silver/master_data/shipments/"

silver_shipments_df = (
    bronze_shipments_df
    # Exclude the CSV header row.
    .filter(~F.col("value").startswith("shipment_id,"))
    # Parse raw CSV text into structured columns.
    .withColumn(
        "parsed",
        F.from_csv(
            F.col("value"),
            shipment_schema,
            {"sep": ",", "mode": "PERMISSIVE"}
        )
    )
    .select("parsed.*", "source_file", "ingested_at")
    # Convert temperature limits to numeric values.
    .withColumn(
        "required_temp_min_c",
        F.col("required_temp_min_c").cast("double")
    )
    .withColumn(
        "required_temp_max_c",
        F.col("required_temp_max_c").cast("double")
    )
)

# Inspect the parsed schema and sample records.
silver_shipments_df.printSchema()
display(silver_shipments_df.limit(5))

root
 |-- shipment_id: string (nullable = true)
 |-- cargo_type: string (nullable = true)
 |-- required_temp_min_c: double (nullable = true)
 |-- required_temp_max_c: double (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



shipment_id,cargo_type,required_temp_min_c,required_temp_max_c,delivery_status,source_file,ingested_at
SHP-00001,general_goods,null,null,planned,shipments.csv,2026-10-26T23:48:02.322Z
SHP-00002,general_goods,null,null,planned,shipments.csv,2026-10-26T23:48:02.322Z
SHP-00003,general_goods,null,null,planned,shipments.csv,2026-10-26T23:48:02.322Z
SHP-00004,general_goods,null,null,planned,shipments.csv,2026-10-26T23:48:02.322Z
SHP-00005,general_goods,null,null,planned,shipments.csv,2026-10-26T23:48:02.322Z


In [0]:
# Validate shipment identifiers and measure missing temperature limits.

display(
    silver_shipments_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("shipment_id").alias("unique_shipment_ids"),
        F.sum(F.when(F.col("shipment_id").isNull(), 1).otherwise(0)).alias("null_shipment_ids"),
        F.sum(F.when(F.col("cargo_type").isNull(), 1).otherwise(0)).alias("null_cargo_types"),
        F.sum(F.when(F.col("delivery_status").isNull(), 1).otherwise(0)).alias("null_delivery_statuses"),
        F.sum(F.when(F.col("required_temp_min_c").isNull(), 1).otherwise(0)).alias("missing_min_temperatures"),
        F.sum(F.when(F.col("required_temp_max_c").isNull(), 1).otherwise(0)).alias("missing_max_temperatures")
    )
)

total_rows,unique_shipment_ids,null_shipment_ids,null_cargo_types,null_delivery_statuses,missing_min_temperatures,missing_max_temperatures
5000,5000,0,0,0,3530,3530


In [0]:
# Save validated shipment records to the Silver Delta layer.

(
    silver_shipments_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_shipments_path)
)

print("Silver shipments transformation completed!")


Silver shipments transformation completed!


## 8. Trip Plans — Parsing and Data Quality

Parse planned trip records from Bronze, standardize timestamps, and validate trip and entity identifiers.

In [0]:
# Inspect a small sample of raw trip plan records from Bronze.

bronze_trip_plans_path = storage_path + "bronze/master_data/trip_plans/"

bronze_trip_plans_df = spark.read.format("delta").load(bronze_trip_plans_path)

display(
    bronze_trip_plans_df
    .select("value")
    .limit(5)
)

value
"trip_id,vehicle_id,driver_id,shipment_id,route_id,planned_start,planned_end,origin,destination,trip_status"
"TRIP-00001-001,VEH-00001,DRV-00001,SHP-00001,RTE-001,2026-10-01T00:00:00Z,2026-10-02T00:00:00Z,Cairo,Cairo Ring,planned"
"TRIP-00002-001,VEH-00002,DRV-00001,SHP-00002,RTE-010,2026-10-01T00:00:00Z,2026-10-02T00:00:00Z,Cairo,North Cairo,planned"
"TRIP-00003-001,VEH-00003,DRV-00001,SHP-00003,RTE-001,2026-10-01T00:00:00Z,2026-10-02T00:00:00Z,Cairo,Cairo Ring,planned"
"TRIP-00004-001,VEH-00004,DRV-00002,SHP-00004,RTE-007,2026-10-01T00:00:00Z,2026-10-02T00:00:00Z,Cairo,Beni Suef,planned"


In [0]:
# Parse trip plans and convert planned timestamps to timestamp types.

trip_plan_schema = """
    trip_id STRING,
    vehicle_id STRING,
    driver_id STRING,
    shipment_id STRING,
    route_id STRING,
    planned_start STRING,
    planned_end STRING,
    origin STRING,
    destination STRING,
    trip_status STRING
"""

silver_trip_plans_path = storage_path + "silver/master_data/trip_plans/"

silver_trip_plans_df = (
    bronze_trip_plans_df
    # Remove the CSV header row.
    .filter(~F.col("value").startswith("trip_id,"))
    # Parse raw CSV records into structured columns.
    .withColumn(
        "parsed",
        F.from_csv(
            F.col("value"),
            trip_plan_schema,
            {"sep": ",", "mode": "PERMISSIVE"}
        )
    )
    .select("parsed.*", "source_file", "ingested_at")
    # Convert planned dates to timestamps.
    .withColumn(
        "planned_start",
        F.to_timestamp("planned_start", "yyyy-MM-dd'T'HH:mm:ssX")
    )
    .withColumn(
        "planned_end",
        F.to_timestamp("planned_end", "yyyy-MM-dd'T'HH:mm:ssX")
    )
)

# Inspect the schema and sample records.
silver_trip_plans_df.printSchema()
display(silver_trip_plans_df.limit(5))

root
 |-- trip_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- planned_start: timestamp (nullable = true)
 |-- planned_end: timestamp (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



trip_id,vehicle_id,driver_id,shipment_id,route_id,planned_start,planned_end,origin,destination,trip_status,source_file,ingested_at
TRIP-00001-001,VEH-00001,DRV-00001,SHP-00001,RTE-001,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,Cairo,Cairo Ring,planned,trip_plans.csv,2026-10-26T23:48:03.440Z
TRIP-00002-001,VEH-00002,DRV-00001,SHP-00002,RTE-010,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,Cairo,North Cairo,planned,trip_plans.csv,2026-10-26T23:48:03.440Z
TRIP-00003-001,VEH-00003,DRV-00001,SHP-00003,RTE-001,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,Cairo,Cairo Ring,planned,trip_plans.csv,2026-10-26T23:48:03.440Z
TRIP-00004-001,VEH-00004,DRV-00002,SHP-00004,RTE-007,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,Cairo,Beni Suef,planned,trip_plans.csv,2026-10-26T23:48:03.440Z
TRIP-00005-001,VEH-00005,DRV-00002,SHP-00005,RTE-007,2026-10-01T00:00:00.000Z,2026-10-02T00:00:00.000Z,Cairo,Beni Suef,planned,trip_plans.csv,2026-10-26T23:48:03.440Z


In [0]:
# Validate trip plan counts, unique IDs, and essential relationship keys.

display(
    silver_trip_plans_df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("trip_id").alias("unique_trip_ids"),
        F.sum(F.when(F.col("trip_id").isNull(), 1).otherwise(0)).alias("null_trip_ids"),
        F.sum(F.when(F.col("vehicle_id").isNull(), 1).otherwise(0)).alias("null_vehicle_ids"),
        F.sum(F.when(F.col("driver_id").isNull(), 1).otherwise(0)).alias("null_driver_ids"),
        F.sum(F.when(F.col("shipment_id").isNull(), 1).otherwise(0)).alias("null_shipment_ids"),
        F.sum(F.when(F.col("route_id").isNull(), 1).otherwise(0)).alias("null_route_ids"),
        F.sum(F.when(F.col("planned_start").isNull(), 1).otherwise(0)).alias("null_planned_starts"),
        F.sum(F.when(F.col("planned_end").isNull(), 1).otherwise(0)).alias("null_planned_ends")
    )
)

total_rows,unique_trip_ids,null_trip_ids,null_vehicle_ids,null_driver_ids,null_shipment_ids,null_route_ids,null_planned_starts,null_planned_ends
5000,5000,0,0,0,0,0,0,0


In [0]:
# Save validated trip plans to the Silver Delta layer.

(
    silver_trip_plans_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_trip_plans_path)
)

print("Silver trip plans transformation completed!")

Silver trip plans transformation completed!


## 9. Silver Layer — Dataset Inventory

Review the Silver Delta datasets created from historical telemetry, anomaly labels, trips, and master data.

In [0]:
# Check that the expected Silver Delta paths can be read and inspect their schemas.

silver_paths = {
    "telemetry": storage_path + "silver/telemetry/",
    "anomaly_labels": storage_path + "silver/anomaly_labels/",
    "trips": storage_path + "silver/trips/",
    "vehicles": storage_path + "silver/master_data/vehicles/",
    "drivers": storage_path + "silver/master_data/drivers/",
    "routes": storage_path + "silver/master_data/routes/",
    "shipments": storage_path + "silver/master_data/shipments/",
    "trip_plans": storage_path + "silver/master_data/trip_plans/"
}

for dataset_name, dataset_path in silver_paths.items():
    # Read Delta metadata and print the available columns without counting all rows.
    dataset_df = spark.read.format("delta").load(dataset_path)
    print(f"{dataset_name}: accessible | columns={len(dataset_df.columns)}")

telemetry: accessible | columns=18
anomaly_labels: accessible | columns=9
trips: accessible | columns=16
vehicles: accessible | columns=9
drivers: accessible | columns=6
routes: accessible | columns=14
shipments: accessible | columns=7
trip_plans: accessible | columns=12


## 10. Referential Integrity — Trip Plans

Validate trip plan references against the Silver vehicle, driver, shipment, and route datasets. Identify invalid references without deleting or modifying source records.

In [0]:
# Check whether trip plan foreign keys exist in their corresponding Silver datasets.

from pyspark.sql import functions as F

# Load the Silver reference datasets.
vehicles_df = spark.read.format("delta").load(silver_paths["vehicles"])
drivers_df = spark.read.format("delta").load(silver_paths["drivers"])
shipments_df = spark.read.format("delta").load(silver_paths["shipments"])
routes_df = spark.read.format("delta").load(silver_paths["routes"])
trip_plans_df = spark.read.format("delta").load(silver_paths["trip_plans"])

# Keep only the keys needed for referential integrity checks.
vehicle_keys = vehicles_df.select("vehicle_id").distinct()
driver_keys = drivers_df.select("driver_id").distinct()
shipment_keys = shipments_df.select("shipment_id").distinct()
route_keys = routes_df.select("route_id").distinct()

# Identify trip plans whose referenced entities do not exist.
invalid_trip_plans_df = (
    trip_plans_df
    .join(vehicle_keys, "vehicle_id", "left_anti")
    .withColumn("invalid_reference", F.lit("vehicle_id"))
)

invalid_driver_refs_df = (
    trip_plans_df
    .join(driver_keys, "driver_id", "left_anti")
    .withColumn("invalid_reference", F.lit("driver_id"))
)

invalid_shipment_refs_df = (
    trip_plans_df
    .join(shipment_keys, "shipment_id", "left_anti")
    .withColumn("invalid_reference", F.lit("shipment_id"))
)

invalid_route_refs_df = (
    trip_plans_df
    .join(route_keys, "route_id", "left_anti")
    .withColumn("invalid_reference", F.lit("route_id"))
)

# Report invalid reference counts without changing the source datasets.
reference_check_results = [
    ("vehicle_id", invalid_trip_plans_df.count()),
    ("driver_id", invalid_driver_refs_df.count()),
    ("shipment_id", invalid_shipment_refs_df.count()),
    ("route_id", invalid_route_refs_df.count())
]

display(
    spark.createDataFrame(
        reference_check_results,
        ["reference_column", "invalid_reference_count"]
    )
)

reference_column,invalid_reference_count
vehicle_id,0
driver_id,0
shipment_id,0
route_id,0


In [0]:
# Compare operational trips with trip plans using trip_id.

trips_df = spark.read.format("delta").load(silver_paths["trips"])

# Select only the fields needed for reconciliation.
trips_compare_df = trips_df.select(
    "trip_id",
    F.col("trip_status").alias("actual_source_status"),
    F.col("planned_end").alias("trips_planned_end")
)

plans_compare_df = trip_plans_df.select(
    "trip_id",
    F.col("trip_status").alias("plan_source_status"),
    F.col("planned_end").alias("plans_planned_end")
)

# Join both sources and flag differences without overwriting either source.
trip_reconciliation_df = (
    trips_compare_df
    .join(plans_compare_df, "trip_id", "full_outer")
    .withColumn(
        "status_mismatch",
        F.col("actual_source_status") != F.col("plan_source_status")
    )
    .withColumn(
        "planned_end_mismatch",
        F.col("trips_planned_end") != F.col("plans_planned_end")
    )
)

# Summarize reconciliation results.
display(
    trip_reconciliation_df.agg(
        F.count("*").alias("total_matched_or_unmatched_trip_ids"),
        F.sum(F.when(F.col("actual_source_status").isNull(), 1).otherwise(0)).alias("missing_from_trips"),
        F.sum(F.when(F.col("plan_source_status").isNull(), 1).otherwise(0)).alias("missing_from_trip_plans"),
        F.sum(F.when(F.col("status_mismatch"), 1).otherwise(0)).alias("status_mismatches"),
        F.sum(F.when(F.col("planned_end_mismatch"), 1).otherwise(0)).alias("planned_end_mismatches")
    )
)

total_matched_or_unmatched_trip_ids,missing_from_trips,missing_from_trip_plans,status_mismatches,planned_end_mismatches
5000,0,0,5000,5000


In [0]:
# Quantify planned end differences and summarize source status combinations.

trip_reconciliation_details_df = (
    trip_reconciliation_df
    .withColumn(
        "planned_end_difference_hours",
        (
            F.col("trips_planned_end").cast("long")
            - F.col("plans_planned_end").cast("long")
        ) / 3600.0
    )
)

# Summarize the time difference distribution.
display(
    trip_reconciliation_details_df.agg(
        F.min("planned_end_difference_hours").alias("min_difference_hours"),
        F.max("planned_end_difference_hours").alias("max_difference_hours"),
        F.avg("planned_end_difference_hours").alias("avg_difference_hours"),
        F.countDistinct("planned_end_difference_hours").alias("distinct_time_differences")
    )
)

# Show the status combinations and their frequencies.
display(
    trip_reconciliation_details_df
    .groupBy("actual_source_status", "plan_source_status")
    .count()
    .orderBy(F.desc("count"))
)

min_difference_hours,max_difference_hours,avg_difference_hours,distinct_time_differences
9.316666666666666,9.316666666666666,9.316666666666178,1


actual_source_status,plan_source_status,count
in_progress,planned,5000



## Data Quality Assessment

### 1. Referential Integrity
All 5,000 trip plans reference existing vehicles, drivers, shipments, and routes. No invalid foreign-key references were found.

### 2. Trip Reconciliation
All 5,000 trip IDs match between `trips.json` and `trip_plans.csv`. However, all records have status mismatches (`in_progress` vs. `planned`) and planned end-time differences of approximately 9 hours and 19 minutes. Both source values are preserved pending investigation.

### 3. Missing Values
Some trip completion fields are null, including actual end time, distance, and fuel consumption. Shipment temperature limits are missing for 3,530 shipments, and planned route distance and duration are missing for all 10 routes. These gaps are documented rather than automatically replaced with assumed values.

### 4. Quality Handling Policy
Invalid records will be quarantined with a recorded reason. Legitimate missing values and source disagreements will be documented separately. No source values will be silently overwritten.


In [0]:

# Run telemetry quality checks in one aggregation to reduce repeated scans.

from pyspark.sql import functions as F

telemetry_df = spark.read.format("delta").load(silver_paths["telemetry"])

telemetry_quality_df = telemetry_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(F.when(F.col("event_id").isNull(), 1).otherwise(0))
        .alias("null_event_ids"),

    F.sum(F.when(F.col("vehicle_id").isNull(), 1).otherwise(0))
        .alias("null_vehicle_ids"),

    F.sum(F.when(F.col("event_timestamp").isNull(), 1).otherwise(0))
        .alias("null_timestamps"),

    F.sum(
        F.when(
            F.col("latitude").isNull()
            | (F.col("latitude") < -90)
            | (F.col("latitude") > 90),
            1
        ).otherwise(0)
    ).alias("invalid_latitudes"),

    F.sum(
        F.when(
            F.col("longitude").isNull()
            | (F.col("longitude") < -180)
            | (F.col("longitude") > 180),
            1
        ).otherwise(0)
    ).alias("invalid_longitudes"),

    F.sum(F.when(F.col("speed_kph") < 0, 1).otherwise(0))
        .alias("negative_speeds"),

    F.sum(F.when(F.col("odometer_km") < 0, 1).otherwise(0))
        .alias("negative_odometer"),

    F.sum(F.when(F.col("rpm") < 0, 1).otherwise(0))
        .alias("negative_rpm"),

    F.sum(F.when(F.col("fuel_rate_lph") < 0, 1).otherwise(0))
        .alias("negative_fuel_rates")
)

display(telemetry_quality_df)


total_rows,null_event_ids,null_vehicle_ids,null_timestamps,invalid_latitudes,invalid_longitudes,negative_speeds,negative_odometer,negative_rpm,negative_fuel_rates
10000000,0,0,0,0,0,0,0,0,0


In [0]:

# Replace null aggregate results with zero when no duplicate groups exist.

duplicate_event_summary_df = (
    telemetry_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_event_id_groups"),
        F.coalesce(F.sum("count"), F.lit(0)).alias("rows_in_duplicate_groups"),
        F.coalesce(
            F.sum(F.col("count") - 1),
            F.lit(0)
        ).alias("excess_duplicate_rows")
    )
)

display(duplicate_event_summary_df)


duplicate_event_id_groups,rows_in_duplicate_groups,excess_duplicate_rows
0,0,0
